# Preamble

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import warnings
import keysight.qcs as qcs
import numpy as np
import matplotlib.pyplot as plt
# import sys
# sys.path.append("..")

from keysight.qcs.experiments import ResonatorSpectroscopy, ResonatorSpectroscopy2D, DispersiveShift
from keysight.qcs.experiments import QubitSpectroscopy,RabiExperiment,T1Experiment,RamseyExperiment,IQDistributionExperiment
from keysight.qcs.analysis import FanoResonance, ComplexLorentzian

from preamble.pump_utils import *
from preamble.mapper_utils import *
from preamble.calibration_utils import *
from preamble.readout_utils import *
from preamble.fitter import *

from IPython.display import clear_output



us,ns,MHz,GHz=1e-6,1e-9,1e6,1e9
pi=np.pi

In [ ]:
cal_file='./config/260702_cal.json'
# set the qubit topology
qubits = qcs.Qudits([0,1,2])
n_qubits=len(qubits)

pairs = [(qubits[i], qubits[i + 1]) for i in range(n_qubits - 1)]
n_mq = len(pairs)

topology = qcs.QuditGraph(qubits, pairs)

# set the local oscillator frequency
LO_freq_readout =  6.1*GHz
LO_freq_qubit = 4.3*GHz

# load the calibration variables
variables_from_json = load_json(cal_file)
variables=make_qcs_var(
            variables_from_json=variables_from_json,
            qubits=qubits,
        )

mapper = qcs.ChannelMapper('localhost')
address = load_address("./config/channel_address.yaml")

channel_mapping(mapper,qubits,address)
set_lo_frequencies_RO(mapper,qubits,address,LO_freq_readout)
set_lo_frequencies_XY(mapper,qubits,address,LO_freq_qubit) 
# set_lo_frequencies_XY(mapper,qubits[0],address,4.1*GHz)

backend = qcs.HclBackend(mapper, fpga_postprocessing=True ,init_time=200*us)

cals = MyCalibrationSet(
        topology=topology,
        channels=mapper.channels,
        variables=variables)
cals.export_values(cal_file)

# shorten the variables name
c=cals.variables


print("Backend is ready:", backend.is_system_ready())


In [ ]:
# set the port of windfreak
# port='socket://163.152.38.107:4000' 
port='COM3'
pump = PumpController(port=port).connect()
pump.set(ch=1,freq=7.52*GHz, power=8.4)
pump.close_all()


# Abort program 

In [ ]:
current_id=backend.get_program_execution_history()[0]['accession_id']
backend.abort_program(current_id)

# Readout calibration

In [ ]:
def set_IQ_program(target_qubit,n_shots=1000):
    amplitudes=qcs.Array('amplitudes',value=[0,c.sx_amp[target_qubit.labels].value],dtype=float)

    ##############################
    IQ_program = qcs.Program()
    IQ_program.add_gate(qcs.GATES.x90, target_qubit)
    IQ_program.add_gate(qcs.GATES.x90, target_qubit)

    IQ_program.add_measurement(target_qubit, new_layer=True)

    IQ_program = qcs.LinkerPass(*cals.linkers.values()).apply(IQ_program)

    IQ_program.n_shots(n_shots)
    IQ_program.sweep(amplitudes, c.sx_amp[target_qubit.labels])
    return IQ_program
    

In [ ]:
def get_classifiction_reference(program,levels=3,plot=False):

    g_prep_IQ=complex_to_array(program.get_iq_array()[0,0])
    e_prep_IQ=complex_to_array(program.get_iq_array()[0,1])

    iq_data = np.concatenate([g_prep_IQ,e_prep_IQ])

    if levels==2:
        centers_init = complex_to_array(np.concatenate([np.mean(program.get_iq_array(),axis=-1)[0]]))
    elif levels==3:
        centers_init = complex_to_array(np.concatenate([np.mean(program.get_iq_array(),axis=-1)[0],[0+0j]]))

    result = fit_gmm_with_centers(
        iq_data,
        centers_init,
        covariance_type="full",
        reg_covar=1e-12,
    )
    classificarion_refs=array_to_complex(result['means'].transpose())

    if levels==2:
        classificarion_refs=np.concatenate([classificarion_refs,[classificarion_refs[1]]])

    if plot==True:
        fig = plot_gmm_result_with_ellipses(result, n_std=2.0)

        fig.show()
        
    return classificarion_refs


In [ ]:
def get_readout_fidelity(program,idx_array=None):

    classifiction_refs=get_classifiction_reference(program,levels=3)

    classifier=qcs.Classifier(references=classifiction_refs)

    if idx_array==None:
        label0=program.get_classified_array(classifier=classifier)[0,0]
        label1=program.get_classified_array(classifier=classifier)[0,1]
    else:
        label0=program.get_classified_array()[0,0][idx_array[0]]
        label1=program.get_classified_array()[0,1][idx_array[1]]

    classified_g=np.unique(label0, return_counts=True)[1]
    classified_e=np.unique(label1, return_counts=True)[1]

    print(classified_g)
    print(classified_e)
    prob_gg=classified_g[0]/np.sum(classified_g[0:2])
    prob_ee=classified_e[1]/np.sum(classified_e[0:2])

    readout_fidelity=(prob_gg+prob_ee)/2
    return readout_fidelity

## Q0

### RO frequency

In [ ]:
target_qubit=qubits[0]
f_center = c.ro_freq[list(target_qubit.labels)].value
f_range = 4 * MHz
n_steps = 31

ro_frequencies=np.linspace(f_center-f_range/2,  f_center+f_range/2, n_steps)

pump.connect().set_on(ch=1)

fidelities=[]
for i, freq in enumerate(ro_frequencies):
    print(f'{i}/{len(ro_frequencies)}')
    c.ro_freq[target_qubit.labels].value=freq
    IQ_program=set_IQ_program(target_qubit,n_shots=1000)    
    IQ_program = qcs.Executor(backend).execute(IQ_program)
    fidelities.append(get_readout_fidelity(IQ_program))
    clear_output()

pump.off(ch=1).close_all()

cals.import_values(cal_file)


In [ ]:
plt.plot(ro_frequencies/GHz,fidelities)
plt.xlabel('RO freq (GHz)')
plt.ylabel('RO fidelity')
plt.axvline(ro_frequencies[np.argmax(fidelities)]/GHz,c='r')

c.ro_freq[target_qubit.labels].value=ro_frequencies[np.argmax(fidelities)]
cals.export_values(cal_file)

print('Optimized RO freq:',round(ro_frequencies[np.argmax(fidelities)]/GHz,4))

### RO amp

In [ ]:
target_qubit=qubits[0]
ro_amp_start = 0.01
ro_amp_end = 0.03
n_steps = 21

ro_amplitudes=np.linspace(ro_amp_start, ro_amp_end, n_steps)

pump.connect().set_on(ch=1)

fidelities=[]
for i, amp in enumerate(ro_amplitudes):
    print(f'{i}/{len(ro_amplitudes)}')
    c.ro_amp[target_qubit.labels].value=amp
    IQ_program=set_IQ_program(target_qubit,n_shots=1000)    
    IQ_program = qcs.Executor(backend).execute(IQ_program)
    fidelities.append(get_readout_fidelity(IQ_program))
    clear_output()

pump.off(ch=1).close_all()

cals.import_values(cal_file)

In [ ]:
plt.plot(ro_amplitudes,fidelities)
plt.axvline(ro_amplitudes[np.argmax(fidelities)],c='r')
plt.xlabel('RO amp (arb.)')
plt.ylabel('RO fidelity')

print('Optimized RO amp:',round(ro_amplitudes[np.argmax(fidelities)],4))

In [ ]:
c.ro_amp[target_qubit.labels].value=ro_amplitudes[np.argmax(fidelities)]
cals.export_values(cal_file)

### RO duration

In [ ]:
target_qubit=qubits[0]
ro_dur_start = 1*us
ro_dur_end = 3*us
n_steps = 31


ro_durations=np.linspace(ro_dur_start,ro_dur_end, n_steps)
pump.connect().set_on(ch=1)

fidelities=[]
for i, dur in enumerate(ro_durations):
    print(f'{i}/{len(ro_durations)}')
    c.ro_dur[target_qubit.labels].value=dur
    c.acq_dur[target_qubit.labels].value=dur
    IQ_program=set_IQ_program(target_qubit,n_shots=1000)    
    IQ_program = qcs.Executor(backend).execute(IQ_program)
    fidelities.append(get_readout_fidelity(IQ_program))
    clear_output()

pump.off(ch=1).close_all()

cals.import_values(cal_file)

In [ ]:
plt.plot(ro_durations/us,fidelities)
plt.axvline(ro_durations[np.argmax(fidelities)]/us,c='r')
plt.xlabel('RO durations (us)')
plt.ylabel('RO fidelity')

print('Optimized RO dur:',round(ro_durations[np.argmax(fidelities)]/us,4),'us')

In [ ]:
c.ro_dur[target_qubit.labels].value=2*us
c.acq_dur[target_qubit.labels].value=2*us
cals.export_values(cal_file)

### IQ distribution plot

In [ ]:
target_qubit=qubits[0]

pump.connect().set_on(ch=1)
IQ_program=set_IQ_program(target_qubit,n_shots=2000)    
IQ_program = qcs.Executor(backend).execute(IQ_program)

pump.off(ch=1).close_all()


In [ ]:
classification_refs=get_classifiction_reference(IQ_program,levels=3,plot=True)

In [ ]:
get_readout_fidelity(IQ_program)

In [ ]:
c.classification_refs[0].value=classification_refs
cals.export_values(cal_file)

### preselection

In [ ]:
target_qubit=qubits[0]

amplitudes=qcs.Array('amplitudes',value=[0,c.sx_amp[target_qubit.labels].value],dtype=float)

##############################
IQ_program = qcs.Program()
IQ_program.add_measurement(target_qubit, new_layer=True)
IQ_program.add_measurement(target_qubit, new_layer=True,pre_delay=1*us)


IQ_program.add_gate(qcs.GATES.x90, target_qubit,pre_delay=1*us)
IQ_program.add_gate(qcs.GATES.x90, target_qubit)

IQ_program.add_measurement(target_qubit, new_layer=True)

IQ_program = qcs.LinkerPass(*cals.linkers.values()).apply(IQ_program)

IQ_program.n_shots(2000)
IQ_program.sweep(amplitudes, c.sx_amp[target_qubit.labels])

##############################

pump.connect().set_on(ch=1)
IQ_program = qcs.Executor(backend).execute(IQ_program)
pump.off(ch=1).close_all()


In [ ]:
c.classification_refs[0].value=get_classifiction_reference(IQ_program,levels=3,plot=True)

In [ ]:
idx0=np.where((IQ_program.get_classified_array(acq_index=0)[0,0]==0) 
              & (IQ_program.get_classified_array(acq_index=1)[0,0]==0))
idx1=np.where((IQ_program.get_classified_array(acq_index=0)[0,1]==0) 
              & (IQ_program.get_classified_array(acq_index=1)[0,1]==0))

idx_array=[idx0,idx1]


In [ ]:
get_readout_fidelity(IQ_program,idx_array=[idx0,idx1])